# 从零开始运行 GraphCast （AutoDL 或者其他新的环境）
-------------------------------------------------------------------
**这是从 https://google-deepmind/graphcast 复现的项目。由 https://github.com/sfsun67 改写和调试。**

**AutoDL 是国内的一家云计算平台，网址是https://www.autodl.com**

你应该有类似的文件结构，这里的数据由 Google Cloud Bucket (https://console.cloud.google.com/storage/browser/dm_graphcast 提供。模型权重、标准化统计和示例输入可在Google Cloud Bucket上找到。完整的模型训练需要下载ERA5数据集，该数据集可从ECMWF获得。
```
.
├── code
│   ├── GraphCast-from-Ground-Zero
│       ├──graphcast
│       ├──tree
│       ├──wrapt
│       ├──graphcast_demo.ipynb
│       ├──README.md
│       ├──setup.py
│       ├──...
├── data
│   ├── dataset
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-01.nc
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-04.nc
│       ├──dataset-source-era5_date-2022-01-01_res-1.0_levels-13_steps-12.nc
│       ├──...
│   ├── params
│       ├──params-GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
│       ├──params-GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
│       ├──...
│   ├── stats
│       ├──stats-mean_by_level.nc
│       ├──...
└────── 
```

PS: 
1. Python 要使用3.10版本。老版本会出现函数调用失效的问题。
2. 你需要仔细核对包的版本，防止出现意外的错误。例如， xarray 只能使用 2023.7.0 版本，其他版本会出现错误。
3. 你需要仔细核对所有包是否安装正确。未安装的包会导致意外错误。例如，tree 和 wrapt 是两个 GraphCast 所必需的包，但是并不在源文件中。例如，tree 和 wrapt 中的 .os 文件未导入，会引发循环调用。他们的原始文件可以在 Colaboratory(https://colab.research.google.com/github/deepmind/graphcast/blob/master/graphcast_demo.ipynb) 的环境中找到。



*代码在如下机器上测试*
1. GPU: TITAN Xp 12GB; CPU: Xeon(R) E5-2680 v4;  JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
2. GPU: V100-SXM2-32GB 32GB; CPU: Xeon(R) Platinum 8255C; JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
3. GPU: RTX 2080 Ti(11GB); CPU: Xeon(R) Platinum 8255C; JAX / 0.3.10 / 3.8(ubuntu18.04) / 11.1
-------------------------------------------------------------------


<p><small><small>版权所有 2023 年 DeepMind Technologies Limited。</small></small></p>
<p><small><small>根据 Apache 许可证第 2.0 版（"许可证"）获得许可；除非符合许可证的规定，否则您不得使用此文件。您可以在 <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a> 获取许可证的副本。</small></small></p>
<p><small><small>除非适用法律要求或书面同意，根据许可证分发的软件是基于 "按原样" 分发的，没有任何明示或暗示的担保或条件。有关许可证下的具体语言，请参见许可证中的权限和限制。</small></small></p>


# 将 Python 版本更新到 3.10.

GraphCast 需要 Python >= 3.10 。推荐 Python 3.10。

在终端中，新建一个名为 GraphCast 的环境。

参考代码如下：
```

# 更新 conda （可选）
conda update -n base -c defaults conda

# 在新环境 GraphCast 中安装 python=3.10  
conda create -n GraphCast python=3.10    

# 更新bashrc中的环境变量
conda init bash && source /root/.bashrc

# 激活新的环境
conda activate GraphCast

# 验证版本
python --version

# 在 Jupyter 中注册 Python 3.10 环境
# 安装 ipykernel 包
conda install ipykernel

# 注册的 Python 3.10 环境的内核名称
python -m ipykernel install --user --name=GraphCast-python3.10
```

注意：Jupyter 注册 Python 3.10 环境后，重启jupyter，使用新的内核 GraphCast-python3.10。

# 安装和初始化


In [1]:
# # 学术资源加速 https://www.autodl.com/docs/network_turbo/  .

# import subprocess
# import os

# result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
# output = result.stdout
# for line in output.splitlines():
#     if '=' in line:
#         var, value = line.split('=', 1)
#         os.environ[var] = value

In [2]:
# # 这一步将使用 shapely 安装环境。为了避免出现ERROR： 无法为 shapely 构建轮子，而安装基于 pyproject.toml 的项目需要轮子。

# !pip uninstall -y shapely
# !conda install -y shapely
# !pip uninstall -y shapely

In [3]:
# # @title Pip 安装 graphcast 和其他依赖项


# %pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

In [4]:
# !pip install xarray

In [5]:
# # @title cartopy 崩溃的解决方法

# !pip uninstall -y shapely
# !pip install shapely --no-binary shapely

In [6]:
# # @title 安装其他依赖项，并解决 xarray 的版本问题。

# # 这里需要将xarray的版本从2023.12.0(2023年12月30日安装)降低到2023.7.0，否则会报错。

# !conda install -y -c conda-forge ipywidgets
# # !pip uninstall -y xarray
# # !pip install xarray==2023.7.0

In [7]:
# pip install ipywidgets

In [8]:
# !pip uninstall -y xarray
# !pip install xarray==2023.7.0

In [9]:
# @title 导入库


import dataclasses
import datetime
import functools
import math
import re
from typing import Optional

import cartopy.crs as ccrs
#from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

import os
# os.environ["JAX_DISABLE_XLA"] = "0"
# XLA_PYTHON_CLIENT_PREALLOCATE=false
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"



def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))


In [10]:
# @title 载入绘图函数

# 这定义了一个名为 select 的函数，它接受一个 xarray.Dataset 对象、一个指定数据集中变量的字符串，以及可选的水平和最大时间步数参数。它返回一个 xarray.Dataset。
def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
#     从数据集中选择与指定变量相对应的数据。
  data = data[variable]
#     如果数据集有一个名为 “batch” 的维度，这行代码选择数据的第一批。
  if "batch" in data.dims:
    data = data.isel(batch=0)
#     如果指定了 max_steps，并且数据集有一个 “time” 维度且步数多于 max_steps，这行代码将数据集限制在前 max_steps 个时间步。
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
#     如果指定了 level 并且它存在于数据集的坐标中，这行代码选择在该特定水平上的数据。
  if level is not None and "level" in data.coords:
    data = data.sel(level=level, method="nearest")
  return data

# 这定义了一个名为 scale 的函数，它接受一个 xarray.Dataset、一个用于缩放的可选中心值和一个表示是否使用鲁棒缩放的布尔值。
# 它返回一个包含数据集、matplotlib.colors.Normalize 对象和颜色映射名称的元组。
def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
#     这些行计算用于归一化的最小和最大值。如果 robust 为 True，它使用第2和第98百分位数来忽略异常值；否则，它使用绝对最小和最大值。
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
# 如果提供了 center 值，这些行调整 vmin 和 vmax 使其与中心等距，确保中心值是颜色刻度的中点。
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
#     函数返回数据集、配置有 vmin 和 vmax 的 Normalize 对象，以及要使用的颜色映射名称（如果有中心点则为 “RdBu_r”，否则为 “viridis”）。
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

# 这定义了一个名为 plot_data 的函数，它接受一个标题和 xarray.Dataset 对象的字典、图形标题、可选的绘图大小、鲁棒缩放标志和子图布局的列数。
# 它返回一个类似于 scale 函数的元组。
def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

# 这些行获取字典中的第一个数据集以确定时间步数（max_steps），并断言所有数据集都有相同数量的时间步。
  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

#     列数设置为指定列数或数据字典长度的最小值。根据列数计算行数。
  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
# 创建一个新的图形，其大小基于行数和列数以及指定的绘图大小。
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
#     设置图形的标题，并调整布局以消除子图之间的空白。
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

#     这个循环为字典中的每个数据集创建子图，设置轴和标题。
  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
#     每个子图使用 imshow 函数显示数据集的第一个时间步，使用指定的归一化和颜色映射。
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
#     为每个子图添加一个颜色条。
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

#     这定义了一个用于动画的 update 函数，它会在每个帧更新标题和数据。
  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

#     使用 FuncAnimation 类创建一个动画，它将为每个帧调用 update 函数。
  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
#     关闭图形（以防止它立即显示），并将动画转换为使用 JavaScript 的 HTML5 视频，然后返回。
  plt.close(figure.number)
  return HTML(ani.to_jshtml())

# 加载数据并初始化模型

## 载入模型参数

选择两种获取模型参数的方式之一：
- **random**：您将获得随机预测，但您可以更改模型架构，这可能会使其运行更快或适应您的设备。
- **checkpoint**：您将获得明智的预测，但受限于模型训练时使用的架构，这可能不适合您的设备。特别是生成梯度会使用大量内存，因此您至少需要25GB的内存（TPUv4或A100）。

检查点在一些方面有所不同：
- 网格大小指定了地球的内部图形表示。较小的网格将运行更快，但输出将更差。网格大小不影响模型的参数数量。
- 分辨率和压力级别的数量必须匹配数据。较低的分辨率和较少的级别会运行得更快。数据分辨率仅影响编码器/解码器。
- 我们的所有模型都预测降水。然而，ERA5包含降水，而HRES不包含。我们标记为 "ERA5" 的模型将降水作为输入，并期望以ERA5数据作为输入，而标记为 "ERA5-HRES" 的模型不以降水作为输入，并专门训练以HRES-fc0作为输入（请参阅下面的数据部分）。

我们提供三个预训练模型：
1. `GraphCast`，用于GraphCast论文的高分辨率模型（0.25度分辨率，37个压力级别），在1979年至2017年间使用ERA5数据进行训练，

2. `GraphCast_small`，GraphCast的较小低分辨率版本（1度分辨率，13个压力级别和较小的网格），在1979年至2015年间使用ERA5数据进行训练，适用于具有较低内存和计算约束的模型运行，

3. `GraphCast_operational`，一个高分辨率模型（0.25度分辨率，13个压力级别），在1979年至2017年使用ERA5数据进行预训练，并在2016年至2021年间使用HRES数据进行微调。此模型可以从HRES数据初始化（不需要降水输入）。


In [11]:
# @title 选择模型
# Rewrite by S.F. Sune, https://github.com/sfsun67.
'''
    我们有三种训练好的模型可供选择, 需要从https://console.cloud.google.com/storage/browser/dm_graphcast准备：
    GraphCast - ERA5 1979-2017 - resolution 0.25 - pressure levels 37 - mesh 2to6 - precipitation input and output.npz
    GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - pressure levels 13 - mesh 2to6 - precipitation output only.npz
    GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz
'''
# 在此路径 /root/data/params 中查找结果，并列出 "params/"中所有文件的名称，去掉名称中的 "params/"perfix。

import os
import glob

# 定义数据目录，请替换成你自己的目录。
dir_path_params = "/root/code/GraphCast-from-Ground-Zero"


# Use glob to get all file paths in the directory
# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
file_paths_params = glob.glob(os.path.join(dir_path_params, "*"))

# 使用glob.glob函数和os.path.join方法获取dir_path_params目录下所有文件的路径。
# Remove the directory path and the ".../params/" prefix from each file name
params_file_options = [os.path.basename(path) for path in file_paths_params]

# 创建一个整数滑块控件random_mesh_size，用于选择网格大小。其默认值为4，最小值为4，最大值为6。较大的网格可以捕获更细的空间特征，但也会增加计算成本。
random_mesh_size = widgets.IntSlider(
    value=4, min=4, max=6, description="Mesh size:")
# 创建一个整数滑块控件random_gnn_msg_steps，用于选择GNN消息传递的步数。其默认值为4，最小值为1，最大值为32。较多的步数能让每个节点接收到来自更远邻居的信息，但会增加计算量和训练时间。
random_gnn_msg_steps = widgets.IntSlider(
    value=8, min=1, max=64, description="GNN message steps:")
# 创建一个下拉菜单控件random_latent_size，用于选择潜在大小。选项是2的4次方到2的9次方，即16到512，其默认值为32。较大的 latent_size 能够捕获更多的信息，但也可能导致过拟合或增加计算成本。
random_latent_size = widgets.Dropdown(
    options=[int(2**i) for i in range(4, 10)], value=32,description="Latent size:")
# 创建一个下拉菜单控件random_levels，用于选择压力水平。选项有13和37，其默认值为13。
random_levels = widgets.Dropdown(
    options=[3, 37], value=3, description="Pressure levels:")

# 创建一个下拉菜单控件params_file，用于选择参数文件。选项是之前从目录中获取的文件名列表。
params_file = widgets.Dropdown(
    options=params_file_options,
    description="Params file:",
    layout={"width": "max-content"})

# 创建一个标签页控件source_tab，其中包含两个标签。第一个标签是一个垂直布局的盒子，包含了前面创建的四个控件。第二个标签是参数文件的下拉菜单。
source_tab = widgets.Tab([
    widgets.VBox([
        random_mesh_size,
        random_gnn_msg_steps,
        random_latent_size,
        random_levels,
    ]),
    params_file,
])
# 为source_tab的两个标签设置标题，分别为"随机参数权重"和"预训练权重"。
source_tab.set_title(0, "随机参数权重（Random）")
source_tab.set_title(1, "预训练权重（Checkpoint）")
# 最后，创建一个垂直布局的盒子，包含了source_tab和一个标签，提示用户运行下一个单元格以加载模型，并告知重新运行该单元格将清除他们的选择。
widgets.VBox([
    source_tab,
    widgets.Label(value="运行下一个单元格以加载模型。重新运行该单元格将清除您的选择。")
])


In [12]:
# @title 加载模型

# 这行代码获取当前选中的标签页的标题，以确定用户选择了哪种参数权重（随机参数权重或预训练权重）。
source = source_tab.get_title(source_tab.selected_index)

# 如果用户选择了“随机参数权重”，则执行以下代码块。
if source == "随机参数权重（Random）":
#     初始化params为None和state为一个空字典。这些将在后面的代码中使用
  params = None  # Filled in below
  state = {}
# 创建一个model_config对象，它包含模型配置的参数。这些参数包括：
# resolution：分辨率，这里设置为0。
# mesh_size：网格大小，取自之前创建的滑块控件的值。
# latent_size：潜在大小，取自下拉菜单控件的值。
# gnn_msg_steps：GNN消息传递步数，取自滑块控件的值。
# hidden_layers：隐藏层的数量，这里设置为1。
# radius_query_fraction_edge_length：查询半径与边长的比例，这里设置为0.6。
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.4)
#     创建一个task_config对象，它包含任务配置的参数。这些参数包括：
# input_variables：输入变量。
# target_variables：目标变量。
# forcing_variables：强迫变量。
# pressure_levels：压力水平，取自下拉菜单控件的值。
# input_duration：输入持续时间。
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
#     如果用户选择了“预训练权重”，则执行以下代码块。
# 这段被注释的代码原本用于从Google Cloud Storage加载预训练权重。
# 现在，它被替换为从本地文件系统加载预训练权重的代码。使用open函数以二进制读取模式打开参数文件，并使用checkpoint.load函数加载检查点。
else:
  assert source == "预训练权重（Checkpoint）"
  '''with gcs_bucket.blob(f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)'''
  
  with open(f"{dir_path_params}/{params_file.value}", "rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
    
#  从检查点中提取参数到params变量，并重新初始化state为一个空字典。 
  params = ckpt.params
  state = {}

# 从检查点中提取模型配置和任务配置。
  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("模型描述:\n", ckpt.description, "\n")
  print("模型许可信息:\n", ckpt.license, "\n")

model_config

ModelConfig(resolution=0, mesh_size=4, latent_size=32, gnn_msg_steps=8, hidden_layers=1, radius_query_fraction_edge_length=0.4, mesh2grid_edge_normalization_factor=None)



## 载入示例数据

有几个示例数据集可用，在几个坐标轴上各不相同：
- **来源**：fake、era5、hres
- **分辨率**：0.25度、1度、6度
- **级别**：13, 37
- **步数**：包含多少个时间步

并非所有组合都可用。
- 由于加载内存的要求，较高分辨率只适用于较少的步数。
- HRES 只有 0.25 度，13 个压力等级。

数据分辨率必须与加载的模型相匹配。

对基础数据集进行了一些转换：
- 我们累积了 6 个小时的降水量，而不是默认的 1 个小时。
- 对于 HRES 数据，每个时间步对应 HRES 在前导时间 0 的预报，实际上提供了 HRES 的 "初始化"。有关详细描述，请参见 GraphCast 论文中的 HRES-fc0。请注意，HRES 无法提供 6 小时的累积降水量，因此我们的模型以 HRES 输入不依赖于降水。但由于我们的模型可以预测降水，因此在示例数据中包含了 ERA5 降水量，以作为地面真实情况的示例。
- 我们在数据中加入了 ERA5 的 "toa_incident_solar_radiation"。我们的模型使用 -6h、0h 和 +6h 辐射作为每 1 步预测的强迫项。在运行中，如果没有现成的 +6h 辐射，可以使用诸如 `pysolar` 等软件包计算辐射。


In [13]:
# import xarray as xr
# import numpy as np

# dir_path_data = "/root/autodl-tmp/"
# # 读取 NetCDF 文件到 example_batch
# input_file_path = f"{dir_path_data}/train_batch.nc"
# example_batch =  xr.open_dataset(input_file_path)
# example_batch.load()

# # 如果你需要立即计算所有数据，可以使用 .load() 或 .compute()
# # example_batch = example_batch.load()  # 或者 example_batch = example_batch.compute()
# # example_batch = example_batch.astype(np.float32)
# # example_batch = example_batch.fillna(0)

# example_batch


In [ ]:
import xarray as xr
import numpy as np

dir_path_data = "/root/autodl-tmp/"
# 读取 NetCDF 文件到 example_batch
input_file_path = f"{dir_path_data}/example_batch_2011_1-6.nc"
example_batch =  xr.open_dataset(input_file_path, chunks={'batch': 1})
example_batch.load()

example_batch


In [ ]:
# train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0,1)) for var in example_batch.data_vars})
# train_batch.load()
# dir_path_data = "/root/autodl-tmp/"
# output_file_path = f"{dir_path_data}/train_batch.nc"  # 保存路径
# train_batch.to_netcdf(output_file_path,mode = 'w')  # 将数据保存为 NetCDF 格式文件
# train_batch

In [ ]:
# 创建一个新的变量 land_sea_mask
import xarray as xr
import numpy as np

# 创建 land_sea_mask 变量，避免中间副本
example_batch["land_sea_mask"] = (
    (~np.isnan(example_batch["so"].isel(time=0, level=0)))
    .astype(np.float32)
)

example_batch = example_batch.fillna(0)
example_batch = example_batch.astype(np.float32)
example_batch

In [ ]:
# import xarray as xr
# import numpy as np

# # 常量
# SEC_PER_DAY = 86400
# _AVG_DAY_PER_YEAR = 365.25

# # 计算进度
# seconds_since_epoch = (example_batch['time'].astype('int64') // 10**9).values
# longitude = example_batch['lon'].values  # 经度

# year_progress = data_utils.get_year_progress(seconds_since_epoch)
# day_progress = data_utils.get_day_progress(seconds_since_epoch, longitude)

# # 生成进度特征
# year_features = data_utils.featurize_progress('year_progress', ['time'], year_progress)
# day_features = data_utils.featurize_progress('day_progress', ['time', 'lon'], day_progress)

# example_batch = example_batch.assign(year_progress_sin=year_features['year_progress_sin'],
#                                year_progress_cos=year_features['year_progress_cos'],
#                                day_progress_sin=day_features['day_progress_sin'],
#                                day_progress_cos=day_features['day_progress_cos'])

# # 创建掩码，排除数值为0的部分
# masked_example_batch = example_batch.where(example_batch != 0)

# # 计算每一天的全图均值和标准差
# daily_mean = masked_example_batch.mean(dim=['lat', 'lon','time','batch'], skipna=True)
# daily_stddev = masked_example_batch.std(dim=['lat', 'lon','time','batch'], skipna=True)

# save_path = '/root/data/stats'
# # 分别保存均值、标准差和标准化值到三个独立的NetCDF文件
# try:
#     daily_mean.to_netcdf(f'{save_path}/stats-mean_by_level.nc', engine='scipy')
#     print("均值文件已保存")
# except RuntimeError as e:
#     print(f"保存均值文件时出错: {e}")

# try:
#     daily_stddev.to_netcdf(f'{save_path}/stats-stddev_by_level.nc', engine='scipy')
#     print("标准差文件已保存")
# except RuntimeError as e:
#     print(f"保存标准差文件时出错: {e}")


# # 去除进度特征
# example_batch = example_batch.drop_vars([
#     'year_progress_sin', 'year_progress_cos',
#     'day_progress_sin', 'day_progress_cos'
# ])
# example_batch

In [ ]:
# @title 加载规范化数据
# Rewrite by S.F. Sune, https://github.com/sfsun67.
# 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
import xarray
import os

dir_path_stats = "/root/data/stats/"

# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
# with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
#   diffs_stddev_by_level = xarray.load_dataset(f).compute()
# 类似于第4行，这行代码打开了另一个文件stats-mean_by_level.nc。
with open(f"{dir_path_stats}/stats-mean_by_level.nc", "rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
# 再次类似于第4行，这行代码打开了第三个文件stats-stddev_by_level.nc。
with open(f"{dir_path_stats}/stats-stddev_by_level.nc", "rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()
# 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()

In [ ]:
diffs_stddev_by_level

In [ ]:
mean_by_level

In [ ]:
stddev_by_level

In [ ]:
# # @title 选择要提取的训练和评估数据

# # 创建了一个整数滑块（IntSlider），用于选择训练步数
# train_steps = widgets.IntSlider(
# #     设置滑块的初始值为1
#     value=1, min=1, max=example_batch.sizes["time"]-2, description="训练步数")
# eval_steps = widgets.IntSlider(
#     value=5, min=1, max=example_batch.sizes["time"]-2, description="评估步数")

# widgets.VBox([
#     train_steps,
#     eval_steps,
#     widgets.Label(value="运行下一个单元格以提取数据。重新运行此单元格将清除您的选择。")
# ])

In [ ]:
# dir_path_data = "/root/autodl-tmp/"
# # 读取 NetCDF 文件到 example_batch
# input_file_path = f"{dir_path_data}/example_batch1.nc"
# # 打开数据集，并使用 chunks 进行延迟加载
# test_batch = xr.open_dataset(input_file_path, chunks={'lat': 200, 'lon': 200})
# test_batch = test_batch.isel(batch=slice(0, 2))
# test_batch.load()
# # 创建一个新的变量 land_sea_mask
# import xarray as xr
# import numpy as np

# # # 延迟加载数据以降低内存占用（如果数据规模较大）
# # example_batch = example_batch.chunk({"lat": 100, "lon": 100})

# # 创建 land_sea_mask 变量，避免中间副本
# test_batch["land_sea_mask"] = (
#     (~np.isnan(test_batch["so"].isel(time=0, level=0)))
#     .astype(np.float32)
#     .persist()  # 在 Dask 中持久化计算结果以减少重复计算
# )



# test_batch = train_batch.fillna(0)
# test_batch = train_batch.astype(np.float32)
# test_batch

In [ ]:
# train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0, 1)) for var in example_batch.data_vars})
# train_batch.load()

# print(train_batch.dims.mapping)

# # # 检查并调整 batch 维度
# # if 'batch' not in train_batch.dims:
# #     for var in train_batch.data_vars:
# #         train_batch[var] = train_batch[var].expand_dims(batch=1)
# # print(train_batch.dims.mapping)

# train_batch

In [ ]:
# # @title 提取训练和评估数据

# print("数据集的维度:", example_batch.dims)

# # 打印数据集的坐标
# print("数据集的坐标:", example_batch.coords)
# train_steps=7

# # 调用了一个名为 data_utils.extract_inputs_targets_forcings 的函数，并将其返回的结果分配给三个变量：train_inputs、train_targets 和 train_forcings。
# train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
# #     example_batch 是一个示例数据批次，target_lead_times 是一个时间切片，用于选择目标数据的时间范围。在这里，我们选择了从24小时到训练步数乘以24小时的时间范围。
#     example_batch, target_lead_times=slice("24h", f"{train_steps*24}h"),
# #     这是函数的第二个参数，它使用 dataclasses.asdict 将 task_config 转换为字典，并将其作为关键字参数传递给函数。
#     **dataclasses.asdict(task_config))

# print("所有示例：  ", example_batch.dims.mapping)
# print("训练输入：  ", train_inputs.dims.mapping)
# print("训练目标： ", train_targets.dims.mapping)
# print("训练强迫：", train_forcings.dims.mapping)

In [ ]:
# import xarray as xr

# # 获取最后一个输入时间步
# last_inputs = {var: train_inputs[var].isel(time=-1) for var in train_inputs.data_vars}

# # 计算残差
# residuals = {}
# for var in train_targets.data_vars:
#     if var in last_inputs:
#         # 计算残差
#         residual = train_targets[var] - last_inputs[var]  # 目标 - 最后输入
#         residuals[var] = residual


# residuals_dataset = xr.Dataset(residuals)

# diffs_stddev_by_level = residuals_dataset.std(dim=['lat', 'lon','time','batch'], skipna=True)

# import jax.numpy as jnp
# save_path = '/root/data/stats'

# try:
#     diffs_stddev_by_level.to_netcdf(f'{save_path}/stats-diffs_stddev_by_level.nc', engine='scipy')
#     print("标准化文件已保存")
# except RuntimeError as e:
#     print(f"保存标准化文件时出错: {e}")

# diffs_stddev_by_level

In [ ]:
# # @title 加载规范化数据
# # Rewrite by S.F. Sune, https://github.com/sfsun67.
# # 定义了一个变量dir_path_stats，它存储了数据统计文件所在的目录路径。
# import xarray as xr
# import os

# dir_path_stats = "/root/data/stats/"

# # 这行代码使用with语句和open函数以二进制读取模式（“rb”）打开一个名为stats-diffs_stddev_by_level.nc的文件。
# with open(f"{dir_path_stats}/stats-diffs_stddev_by_level.nc", "rb") as f:
#   diffs_stddev_by_level = xarray.load_dataset(f).compute()

In [ ]:
# from jax import device_put
# from jax import devices
# from jax.sharding import Mesh, PartitionSpec as P
# from jax.experimental import mesh_utils
# from jax.experimental.shard_map import shard_map

# # 定义构建和包装GraphCast预测器的函数
# def construct_wrapped_graphcast(
#     model_config: graphcast.ModelConfig,
#     task_config: graphcast.TaskConfig):
#   """Constructs and wraps the GraphCast Predictor."""
#   # 创建一个更深层次的一步预测器
#   predictor = graphcast.GraphCast(model_config, task_config)

#   # 修改输入/输出以处理从float32到BFloat16的转换
#   # predictor = casting.Bfloat16Cast(predictor)

#   # 在应用输入/目标的规范化之后，进行BFloat16的转换
#   predictor = normalization.InputsAndResiduals(
#       predictor,
#       diffs_stddev_by_level=diffs_stddev_by_level,
#       mean_by_level=mean_by_level,
#       stddev_by_level=stddev_by_level)

#   # 包装所有内容，使一步模型能够产生轨迹
#   predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
#   return predictor

# # 定义前向运算函数
# @hk.transform_with_state
# def run_forward(model_config, task_config, inputs, targets_template, forcings):
#   predictor = construct_wrapped_graphcast(model_config, task_config)
#   return predictor(inputs, targets_template=targets_template, forcings=forcings)

# # 定义计算损失函数的函数
# @hk.transform_with_state
# def loss_fn(model_config, task_config, inputs, targets, forcings):
#   predictor = construct_wrapped_graphcast(model_config, task_config)
#   loss, diagnostics = predictor.loss(inputs, targets, forcings)
#   return xarray_tree.map_structure(
#       lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
#       (loss, diagnostics))

# # 定义计算梯度的函数
# def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
#   def _aux(params, state, i, t, f):
#     (loss, diagnostics), next_state = loss_fn.apply(
#         params, state, jax.random.PRNGKey(0), model_config, task_config,
#         i, t, f)
#     return loss, (diagnostics, next_state)
#   (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
#       _aux, has_aux=True)(params, state, inputs, targets, forcings)
#   return loss, diagnostics, next_state, grads

# # 定义一个函数，用于通过functools.partial传递配置
# def with_configs(fn):
#   return functools.partial(
#       fn, model_config=model_config, task_config=task_config)

# # 定义一个函数，用于通过functools.partial传递参数和状态
# def with_params(fn):
#   return functools.partial(fn, params=params, state=state)

# # 定义一个函数，用于丢弃状态并只返回预测结果
# def drop_state(fn):
#   return lambda **kw: fn(**kw)[0]

# # 使用jax.jit编译初始化函数
# init_jitted = jax.jit(with_configs(run_forward.init))


# # 编译损失函数和梯度函数
# loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
# grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
# run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
#     run_forward.apply))))


In [ ]:
# @title Build jitted functions, and possibly initialize random weights
# Construct the model and initialize the weights.
# 构建模型并初始化权重

# 模型组网
# Construct the model
def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # Deeper one-step predictor.
  predictor = graphcast.GraphCast(model_config, task_config)

  # Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to
  # from/to float32 to/from BFloat16.
  # predictor = casting.Bfloat16Cast(predictor)

  # Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from
  # BFloat16 happens after applying normalization to the inputs/targets.
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # Wraps everything so the one-step model can produce trajectories.
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor

# 前向运算
# forward
@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

# 计算损失函数
# loss function
@hk.transform_with_state    # used to convert a pure function into a stateful function
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)    # constructs and wraps a GraphCast Predictor, which is a model used for making predictions in a graph-based machine learning task.
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

# 计算梯度
# gradient
def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), model_config, task_config,
        i, t, f)
    return loss, (diagnostics, next_state)
  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads

# Jax doesn't seem to like passing configs as args through the jit. Passing it
# in via partial (instead of capture by closure) forces jax to invalidate the
# jit cache if you change configs.
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# Always pass params and state, so the usage below are simpler
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# Our models aren't stateful, so the state is always empty, so just return the
# predictions. This is requiredy by our rollout code, and generally simpler.
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

init_jitted = jax.jit(with_configs(run_forward.init))

# if params is None:
#   params, state = init_jitted(
#       rng=jax.random.PRNGKey(0),
#       inputs=train_inputs,
#       targets_template=train_targets,
#       forcings=train_forcings)

loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
    run_forward.apply))))

In [ ]:
# import numpy as np

# # 假设 'params' 是你训练后的模型参数
# # 你可以通过以下代码保存到 .npz 文件

# def save_params_to_npz(params, filepath):
#     # 将params的内容转换为字典结构，便于保存
#     params_dict = {k: v for k, v in params.items()}
    
#     # 使用np.savez将参数保存到指定的文件中
#     np.savez(filepath, **params_dict)
#     print(f"模型参数已保存到 {filepath}")

# # 定义文件保存路径
# save_filepath = "model_params_test.npz"

# # 调用保存函数，将参数保存
# save_params_to_npz(params, save_filepath)

# 训练模型

以下操作需要大量内存，而且根据所使用的加速器，只能在低分辨率数据上拟合很小的 "随机 "模型。它使用上面选择的训练步数。

第一次执行单元需要更多时间，因为其中包括函数的 jit 时间。

In [ ]:
import optax  # 导入 optax 库
import jax
import jax.numpy as jnp
from jax import random
import numpy as np
import gc  # 导入垃圾回收模块
from jax import device_put
from jax import devices


def release_memory(*var_names):
    for var_name in var_names:
        if var_name in globals():  # 检查变量是否存在于全局命名空间
            del globals()[var_name]
    gc.collect()

release_memory("train_batch")

# 初始化训练步长
train_steps = 1  # 初始自回归预测步长

# 定义批次大小和总训练轮数
batch_size = example_batch.dims['batch']  # 根据你的数据调整
total_epochs = 100  # 总训练轮数
learning_rate = 0.0001
# 定义优化器
optimizer = optax.adam(learning_rate=learning_rate)
opt_state = optimizer.init(params)
    
# 训练循环
for epoch in range(total_epochs):
    # 每20轮增加1个时间步长
    if epoch % 10 == 0 and epoch > 0:
        train_steps += 1
        print(f"训练步长增加: 当前 train_steps = {train_steps}")
    print(f"当前 Epoch: {epoch}, Train Steps: {train_steps}")
    
    for start_idx in range(batch_size):
        train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(start_idx, start_idx + 1)) for var in example_batch.data_vars})
        # train_batch = xr.Dataset({var: example_batch[var].isel(batch=slice(0,1)) for var in example_batch.data_vars})
        train_batch.load()
        print(train_batch.dims.mapping)
        
        train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
            train_batch,
            target_lead_times=slice("24h", f"{train_steps * 24}h"),
            **dataclasses.asdict(task_config))
        print(train_inputs.dims.mapping)
        # 打印train_inputs的维度和数据类型
        print("train_inputs变量维度：")
        for var_name, var_data in train_inputs.items():
            print(f"Variable: {var_name}, Shape: {var_data.shape}, dtype: {var_data.dtype}")
        
        # 如果参数为空，则初始化参数和状态
        if params is None:
          params, state = init_jitted(
              rng=jax.random.PRNGKey(0),
              inputs=train_inputs,
              targets_template=train_targets,
              forcings=train_forcings)
        
        # 在每个批次中使用不同的随机数种子
        rng = random.PRNGKey(epoch * total_epochs + start_idx)
        print("batch start")
        
        loss, diagnostics, next_state, grads = grads_fn_jitted(
            params=params,
            state=state,
            inputs=train_inputs,
            targets=train_targets,
            forcings=train_forcings
        )
        print("compute grads")

        # 使用优化器更新参数
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)  # 应用更新到参数
        release_memory("train_batch")
        
     # 打印损失值   
    mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])   
    print(f"Epoch {epoch}: Loss: {loss:.4f} :Mean |grad|: {mean_grad:.6f}")

In [ ]:
# import xarray as xr
# import numpy as np
# import os
# import glob
# import optax
# import jax
# import jax.numpy as jnp
# from jax import random
# import gc

# # 初始化训练步长
# train_steps = 1  # 初始自回归预测步长

# # 总训练轮数
# total_epochs = 200  # 总训练轮数
# learning_rate = 0.00005

# # 优化器
# optimizer = optax.adam(learning_rate=learning_rate)
# opt_state = optimizer.init(params)

# # 文件路径
# dir_path_data = "/root/autodl-tmp/"
# save_path = "/root/data/stats/"

# # 获取所有 NetCDF 文件路径
# input_files = glob.glob(f"{dir_path_data}/example_batch*.nc")

# def release_memory(*var_names):
#     """释放内存"""
#     for var_name in var_names:
#         if var_name in globals():
#             del globals()[var_name]
#     gc.collect()

# # 训练循环
# for epoch in range(total_epochs):
#     if epoch % 20 == 0 and epoch > 0:
#         train_steps += 1
#         print(f"训练步长增加: 当前 train_steps = {train_steps}")

#     print(f"当前 Epoch: {epoch}, Train Steps: {train_steps}")

#     # 逐个文件读取并训练
#     for file_path in input_files:
#         print(f"正在处理文件：{file_path}")

#         # 读取当前文件的 dataset
#         example_batch = xr.open_dataset(file_path, chunks={'batch': 5})
        
#         # 创建 land_sea_mask 变量，避免中间副本
#         example_batch["land_sea_mask"] = (
#             (~np.isnan(example_batch["so"].isel(time=0, level=0)))
#             .astype(np.float32)
#             .persist()  # 在 Dask 中持久化计算结果以减少重复计算
#         )
        
#         example_batch = example_batch.fillna(0)
#         example_batch = example_batch.astype(np.float32)

#         batch_size = example_batch.dims['batch']  # 批次大小
#         batch_one = 1

#         # 遍历当前文件中的所有批次
#         for start_idx in range(0, batch_size, batch_one):
#             end_idx = start_idx + batch_size
#             # 读取当前批次
#             train_batch = xr.Dataset({
#                 var: example_batch[var].isel(batch=slice(start_idx, end_idx))
#                 for var in example_batch.data_vars
#             })
#             train_batch.load()  # 确保数据加载到内存中

#             # 提取输入、目标和强迫
#             train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
#                 train_batch,
#                 target_lead_times=slice("24h", f"{train_steps * 24}h"),
#                 **dataclasses.asdict(task_config)
#             )

#             # 使用不同的随机种子
#             rng = random.PRNGKey(epoch * total_epochs + start_idx)

#             # # 计算损失和梯度
#             # loss, diagnostics = loss_fn_jitted(
#             #     rng=rng,
#             #     inputs=train_inputs,
#             #     targets=train_targets,
#             #     forcings=train_forcings
#             # )

#             loss, diagnostics, next_state, grads = grads_fn_jitted(
#                 params=params,
#                 state=state,
#                 inputs=train_inputs,
#                 targets=train_targets,
#                 forcings=train_forcings
#             )

#             # 计算梯度的平均值
#             mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])

#             # 更新参数
#             updates, opt_state = optimizer.update(grads, opt_state, params)
#             params = optax.apply_updates(params, updates)  # 应用更新

#            # 打印损失值   
#     mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])   
#     print(f"Epoch {epoch}: Loss: {loss:.4f} :Mean |grad|: {mean_grad:.6f}")

In [ ]:
import numpy as np

# 假设 'params' 是你训练后的模型参数
# 你可以通过以下代码保存到 .npz 文件

def save_params_to_npz(params, filepath):
    # 将params的内容转换为字典结构，便于保存
    params_dict = {k: v for k, v in params.items()}
    
    # 使用np.savez将参数保存到指定的文件中
    np.savez(filepath, **params_dict)
    print(f"模型参数已保存到 {filepath}")

# 定义文件保存路径
save_filepath = "model_params_all_2.npz"

# 调用保存函数，将参数保存
save_params_to_npz(params, save_filepath)